## Autoencoder on  2d Müller-Brown potential



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.cm as cm
import bisect

### 2d Müller-Brown potential

We define the potential $V$ as a class and a function for sampling the trajectory of the Brownian dynamics

$$
dX_t = - \nabla V(X_t) dt + \sqrt{2\beta^{-1}} dW_t
$$

In [ ]:
# Mueller-Brown potential $V(x)$ in 2d
class MuellerPotential:
    def __init__(self, *argv):
        
        # Parameters in the definition of V
        self.a = [-1, -1, -6.5, 0.7]
        self.b = [0, 0, 11, 0.6]
        self.c = [-10, -10, -6.5, 0.7]
        self.A = [-200, -100, -170, 15]
        self.xc = [1, 0, -0.5, -1]
        self.yc = [0, 0.5, 1.5, 1]

        self.x_domain = [-1.8, 1.2]
        self.y_domain = [-0.5, 2.2]
        self.v_min_max = [-130, 20]
        self.contour_levels = [-130, -100, -80, -60, -40, -20, 0.0]
        self.density_max = 0.35
        
    # the potential    
    def V(self, x):
        s = 0
        for i in range(4):
            dx = x[0] - self.xc[i]
            dy = x[1] - self.yc[i]
            s += self.A[i] * np.exp(self.a[i] * dx**2 + self.b[i] * dx * dy + self.c[i] * dy**2)
        return s
    
    # gradient of the potential    
    def gradV(self, x):
        s = 0
        dVx = 0
        dVy = 0
        for i in range(4):
            dx = x[0] - self.xc[i]
            dy = x[1] - self.yc[i]            
            dVx += self.A[i] * (2 * self.a[i] * dx + self.b[i] * dy) * np.exp(self.a[i] * dx**2 + self.b[i] * dx * dy + self.c[i] * dy**2)
            dVy += self.A[i] * (self.b[i] * dx + 2 * self.c[i] * dy) * np.exp(self.a[i] * dx**2 + self.b[i] * dx * dy + self.c[i] * dy**2)
        return np.array((dVx, dVy))

# sample the SDE using Euler-Maruyama scheme

def sample(pot, beta=1.0, delta_t = 0.001, N=10000, seed=42):
    rng = np.random.default_rng(seed=seed)
     
    X = [-0.6, 1.2]
    dim = 2 
    traj = []
    save = 100
    tlist = []
    for i in tqdm(range(N)):
        b = rng.normal(size=(dim,))
        X = X - pot.gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b
        if i % save==0:
            traj.append(X)
            tlist.append(i * delta_t)

    return np.array(tlist), np.array(traj)

### define an object of the potential class

In [ ]:
pot = MuellerPotential()  

### discrete the 2d space into a grid 

In [ ]:
nx = 100
ny = 150

dx = (pot.x_domain[1] - pot.x_domain[0]) / nx
dy = (pot.y_domain[1] - pot.y_domain[0]) / ny

gridx = np.linspace(pot.x_domain[0], pot.x_domain[1], nx)
gridy = np.linspace(pot.y_domain[0], pot.y_domain[1], ny)
x_plot = np.outer(gridx, np.ones(ny)) 
y_plot = np.outer(gridy, np.ones(nx)).T 

# get grid points
x2d = np.concatenate((x_plot.reshape(nx * ny, 1), y_plot.reshape(nx * ny, 1)), axis=1)

# compute the potential $V$ at grid points
pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)

### visualize the potential

The potenial has two deep local minimum points, which are separated by a shallow local minimum point.

In [ ]:
fig, ax = plt.subplots()

# visualize the potential and its contour lines
im = ax.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm', vmin=pot.v_min_max[0], vmax=pot.v_min_max[1])
contours = ax.contour(x_plot, y_plot, pot_on_grid,  pot.contour_levels)

ax.clabel(contours, inline=True, fontsize=13,colors='black')

ax.set_aspect('equal')
ax.tick_params(axis='both', labelsize=15)

ax.set_xticks([-1.5, -1.0, -0.5, 0, 0.5, 1.0])
ax.set_yticks([-0.5, 0, 0.5, 1.0, 1.5, 2.0])
ax.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax.set_ylim([pot.y_domain[0], pot.y_domain[1]])

ax.set_title("Müller-Brown potential",fontsize=20)
cbar = fig.colorbar(im, ax=ax, shrink=1.0)
cbar.ax.tick_params(labelsize=15)

### generate a long trajectory of the Brownian dynamics

$$
dX_t = - \nabla V(X_t) dt + \sqrt{2\beta^{-1}} dW_t
$$

We will use the trajectory data to train the autoencoder.

In [ ]:
tlist, trajectory = sample(pot, beta=0.1, delta_t=0.0002, N=1000000)

print ('shape of the trajectory data:', trajectory.shape)

### plot the trajectory data

In [ ]:
fig = plt.figure(figsize=(12,3))

ax1 = fig.add_subplot(1, 3, 1)
ax2 = fig.add_subplot(1, 3, 2)
ax3 = fig.add_subplot(1, 3, 3)

# evaluate potential on grid points
pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)
# plot contour lines of the potential
contours = ax1.contour(x_plot, y_plot, pot_on_grid, levels=pot.contour_levels, cmap='coolwarm')

# scatter plot of the trajectory data
ax1.scatter(trajectory[:,0], trajectory[:,1], alpha=0.5, c='k', s=4)

ax1.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax1.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax1.set_title('trajectory')

# plot time-series of the x component
ax2.plot(tlist, trajectory[:,0])
ax2.set_ylim([pot.x_domain[0], pot.x_domain[1]])
ax2.set_title('x coodinate along trajectory')

# plot time-series of the y component
ax3.plot(tlist, trajectory[:,1])
ax3.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax3.set_title('y coodinate along trajectory')

plt.show()

In [ ]:
    
def find_MEP(pot, N, min_A, min_B, dt=0.0001, NSteps=1000, with_reparam=True):

    line = np.array(np.linspace(min_A, min_B, N)) 
    
    delta_t = dt

    for istep in range(NSteps):
        
        # Update 
        for idx in range(N) :
            X = line[idx, :]
            line[idx,:] = X - pot.gradV(X) * delta_t 

        # Compute distances
        dists = [0.0]
        for idx in range(N-1) :
            dists.append( np.linalg.norm(line[idx,:] - line[idx+1,:]) + dists[idx] )

        dists = dists / dists[-1]

        # Reparametrization

        new_states = [line[0,:]]

        for idx in range(N-2):
            r = (idx+1) / (N - 1)
            pos = bisect.bisect_left(dists, r)
            t = (r-dists[pos-1]) / (dists[pos] - dists[pos-1])
            if t < 0 or t > 1 :
                print ("Wrong:", t)
                
            new_states.append((1-t) * line[pos-1, :] + t * line[pos, :])

        new_states.append(line[-1,:])
        
        if with_reparam : 
            line = np.array(new_states)

    return line

In [ ]:
def plot_potential_on_axis(ax):
    
    # visualize the potential and its contour lines
    im = ax.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm', vmin=pot.v_min_max[0], vmax=pot.v_min_max[1])
    contours = ax.contour(x_plot, y_plot, pot_on_grid,  pot.contour_levels)

    #ax.clabel(contours, inline=True, fontsize=13,colors='black')

    ax.set_aspect('equal')
    ax.tick_params(axis='both', labelsize=10)

    ax.set_xticks([-1.5, -1.0, -0.5, 0, 0.5, 1.0])
    ax.set_yticks([-0.5, 0, 0.5, 1.0, 1.5, 2.0])
    ax.set_xlim([pot.x_domain[0], pot.x_domain[1]])
    ax.set_ylim([pot.y_domain[0], pot.y_domain[1]])    
        
fig = plt.figure(figsize=(9,9))

ax1 = fig.add_subplot(2, 2, 1)
ax2 = fig.add_subplot(2, 2, 2)
ax3 = fig.add_subplot(2, 2, 3)
ax4 = fig.add_subplot(2, 2, 4)

xa = [-0.5, 1.5]
xb = [0.6, 0.0]

path = find_MEP(pot, 30, xa, xb, NSteps=0)
plot_potential_on_axis(ax=ax1)
ax1.scatter(path[:,0], path[:,1], c='k')
ax1.set_title('step 0')

path = find_MEP(pot, 30, xa, xb, NSteps=20)
plot_potential_on_axis(ax=ax2)
ax2.scatter(path[:,0], path[:,1], c='k')
ax2.set_title('step 20')

path = find_MEP(pot, 30, xa, xb, NSteps=60)
plot_potential_on_axis(ax=ax3)
ax3.scatter(path[:,0], path[:,1], c='k')
ax3.set_title('step 60')

path = find_MEP(pot, 30, xa, xb, NSteps=100)
plot_potential_on_axis(ax=ax4)
ax4.scatter(path[:,0], path[:,1], c='k')
ax4.set_title('step 100')

plt.show()

In [ ]:
fig = plt.figure(figsize=(9,9))

ax1 = fig.add_subplot(2, 2, 1)
ax2 = fig.add_subplot(2, 2, 2)
ax3 = fig.add_subplot(2, 2, 3)
ax4 = fig.add_subplot(2, 2, 4)

xa = [-0.5, 1.5]
xb = [0.6, 0.0]

path = find_MEP(pot, 30, xa, xb, NSteps=0, with_reparam=False)
plot_potential_on_axis(ax=ax1)
ax1.scatter(path[:,0], path[:,1], c='k')
ax1.set_title('step 0')

path = find_MEP(pot, 30, xa, xb, NSteps=20, with_reparam=False)
plot_potential_on_axis(ax=ax2)
ax2.scatter(path[:,0], path[:,1], c='k')
ax2.set_title('step 20')

path = find_MEP(pot, 30, xa, xb, NSteps=60, with_reparam=False)
plot_potential_on_axis(ax=ax3)
ax3.scatter(path[:,0], path[:,1], c='k')
ax3.set_title('step 60')

path = find_MEP(pot, 30, xa, xb, NSteps=100, with_reparam=False)
plot_potential_on_axis(ax=ax4)
ax4.scatter(path[:,0], path[:,1], c='k')
ax4.set_title('step 100')

plt.show()

In [ ]:
pot_on_line = [pot.V(state) for state in path]

max_idx = np.argmax(pot_on_line)
X = path[max_idx,:]

tau = path[max_idx+1] - path[max_idx-1]
tau = tau / np.linalg.norm(tau)

print ('tangent direction:', tau)

In [ ]:
xa = [-0.5, 1.5]
xb = [0.6, 0.0]

path = find_MEP(pot, 30, xa, xb, NSteps=200)
pot_on_line = [pot.V(state) for state in path]
max_idx = np.argmax(pot_on_line)

fig = plt.figure(figsize=(9,4))
ax1 = fig.add_subplot(1, 2, 1)

plot_potential_on_axis(ax=ax1)
ax1.scatter(path[:,0], path[:,1], c='k')
ax1.set_title('MEP')
ax1.plot(path[max_idx,0], path[max_idx,1], marker='o', c='w', markersize=12)

ax2 = fig.add_subplot(1, 2, 2)

ax2.plot(pot_on_line)
ax2.set_title('potential along MEP')
plt.show()


In [ ]:
X = path[max_idx,:]
X_list = [X]
N = 1000
delta_t = 0.0001

for idx in range(N) :
    X = X - pot.gradV(X) * delta_t 
    X_list.append(X)

In [ ]:
fig, ax = plt.subplots()

plot_potential_on_axis(ax=ax)

X_traj = np.array(X_list)
ax.scatter(X_traj[::5,0], X_traj[::5,1])

ax.plot(X_traj[0,0], X_traj[0,1], marker='o', c='w', markersize=12)
ax.plot(X_traj[-1,0], X_traj[-1,1], marker='X', c='k', markersize=6)

plt.show()

In [ ]:
X_list = [X]
N = 1000
delta_t = 0.0001

for idx in range(N) :
    drift = pot.gradV(X)
    X = X + (- drift + 2.0 * np.dot(drift, tau) * tau) * delta_t
    X_list.append(X)

In [ ]:
fig, ax = plt.subplots()

plot_potential_on_axis(ax=ax)

X_traj = np.array(X_list)
ax.scatter(X_traj[::5,0], X_traj[::5,1])

ax.plot(X_traj[0,0], X_traj[0,1], marker='o', c='w', markersize=12)
ax.plot(X_traj[-1,0], X_traj[-1,1], marker='X', c='k', markersize=6)

plt.show()